# SpaceX Falcon 9 Exploratory Data Analysis with Visualization

This notebook explores SpaceX Falcon 9 launch data using Python data visualization libraries.

The analysis focuses on launch sites, mission outcomes, yearly launch trends, payload information, and relationships between launch variables.

## 1. Import Libraries

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load SpaceX Launch Data

In [ ]:
response = requests.get("https://api.spacexdata.com/v4/launches/past")
response.raise_for_status()

launches = response.json()
df = pd.json_normalize(launches)

rocket_response = requests.get("https://api.spacexdata.com/v4/rockets")
rocket_response.raise_for_status()

rockets = rocket_response.json()
rocket_lookup = {rocket["id"]: rocket["name"] for rocket in rockets}
    
    df["rocket_name"] = df["rocket"].map(rocket_lookup)
    df = df[df["rocket_name"].str.contains("Falcon 9", na=False)].copy()
    
    df["date_utc"] = pd.to_datetime(df["date_utc"], errors="coerce")
    df["year"] = df["date_utc"].dt.year
    
    print("Falcon 9 launches:", len(df))
    df.head()

## 3. Launches by Year

In [ ]:
yearly_launches = df.groupby("year").size().reset_index(name="launch_count")
    
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=yearly_launches, x="year", y="launch_count", marker="o")
    plt.title("SpaceX Falcon 9 Launches by Year")
    plt.xlabel("Year")
    plt.ylabel("Number of Launches")
    plt.tight_layout()
    plt.show()

## 4. Launches by Launch Site

In [ ]:
launch_site_counts = df["launchpad"].value_counts().reset_index()
    launch_site_counts.columns = ["launch_site", "launch_count"]
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=launch_site_counts, x="launch_count", y="launch_site")
    plt.title("Falcon 9 Launches by Launch Site")
    plt.xlabel("Number of Launches")
    plt.ylabel("Launch Site")
    plt.tight_layout()
    plt.show()

## 5. Mission Success and Failure

In [ ]:
outcome_counts = df["success"].map({True: "Success", False: "Failure"}).value_counts()
    
    plt.figure(figsize=(7, 5))
    sns.barplot(x=outcome_counts.index, y=outcome_counts.values)
    plt.title("Falcon 9 Mission Outcomes")
    plt.xlabel("Mission Outcome")
    plt.ylabel("Number of Launches")
    plt.tight_layout()
    plt.show()

## 6. Mission Outcome by Launch Site

In [ ]:
site_outcomes = pd.crosstab(
    df["launchpad"],
    df["success"].map({True: "Success", False: "Failure"})
    )
    
    site_outcomes.plot(kind="bar", figsize=(10, 5))
    plt.title("Mission Outcomes by Launch Site")
    plt.xlabel("Launch Site")
    plt.ylabel("Number of Launches")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 7. Success Rate by Year

In [ ]:
success_by_year = df.groupby("year")["success"].mean().reset_index()
    success_by_year["success_rate"] = success_by_year["success"] * 100
    
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=success_by_year, x="year", y="success_rate", marker="o")
    plt.title("Falcon 9 Success Rate by Year")
    plt.xlabel("Year")
    plt.ylabel("Success Rate (%)")
    plt.tight_layout()
    plt.show()

## 8. Orbit Distribution

In [ ]:
if "orbit" in df.columns:
    orbit_counts = df["orbit"].value_counts().head(10)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x=orbit_counts.values, y=orbit_counts.index)
    plt.title("Most Common Falcon 9 Orbit Types")
    plt.xlabel("Number of Launches")
    plt.ylabel("Orbit")
    plt.tight_layout()
    plt.show()

## 9. Exploratory Summary

In [ ]:
print("Total Falcon 9 launches:", len(df))
print("Successful launches:", int(df["success"].sum()))
print("Failed launches:", int((df["success"] == False).sum()))
print("Years covered:", df["year"].min(), "to", df["year"].max())

## Conclusion

The visualization analysis provides an overview of Falcon 9 launch activity, launch-site usage, mission outcomes, yearly trends, and orbit distributions.